This code makes a FHIR Patient search request to retrieve a patient record using an NHS number identifier. Here’s a clear, structured description of what the request is doing.

High-level description

The request performs an HTTP GET against a FHIR server’s Patient endpoint, searching for a patient whose identifier matches a given NHS number.
Authentication is handled using HTTP Basic Auth, with credentials loaded from environment variables.

Breakdown of the FHIR request
1. Endpoint being called
GET {FHIR_SERVER}/Patient?identifier=https://fhir.nhs.uk/Id/nhs-number|9999999522


Resource type: Patient

Operation: Search

Search parameter: identifier

2. Search parameter details
identifier=https://fhir.nhs.uk/Id/nhs-number|9999999522


This follows the FHIR search syntax for identifiers:

identifier = system | value


system: https://fhir.nhs.uk/Id/nhs-number
Identifies the namespace for NHS numbers (UK-specific FHIR identifier system)

value: 9999999522
The NHS number being searched for

👉 This tells the FHIR server:

“Return any Patient resource that has an identifier with this system and value.”

In [15]:
import requests
import fhirclient.models.patient as pat
from requests.auth import HTTPBasicAuth
from dotenv import load_dotenv
load_dotenv()
import os
import json

fhir_password = os.getenv("FHIR_PASSWORD")
fhir_username = os.getenv("FHIR_USERNAME")
server = os.getenv("FHIR_SERVER")

nhsNumber = os.getenv("NHS_NUMBER")
if nhsNumber == None:
    nhsNumber = '9737873858'

api_url = server + "Patient?identifier=https://fhir.nhs.uk/Id/nhs-number|"+nhsNumber
print("Calling GET "+api_url)
response = requests.get(api_url, auth=HTTPBasicAuth(fhir_username, fhir_password))
patientJSON = response.json()
#print(patientJSON)
print(json.dumps(patientJSON, indent=4))
patientId = None

if ((patientJSON['total']> 0) and (len(patientJSON['entry'])>0)):
    patientId = patientJSON['entry'][0]['resource']['id']
    patient = pat.Patient(patientJSON['entry'][0]['resource'])
    print("Patient Id: " + patientId)
    print("Date of Birth: "+patient.birthDate.isostring)
    for id in patient.identifier:
        print('Identifier: '+ id.value + " " + id.type.coding[0].code)



Calling GET http://192.168.1.62/healthconnect/cdr/fhir/r4/Patient?identifier=https://fhir.nhs.uk/Id/nhs-number|9737873858
{
    "resourceType": "Bundle",
    "id": "4cc970fa-f6bb-4188-813b-63dc6592564a",
    "type": "searchset",
    "timestamp": "2026-07-22T10:47:38Z",
    "total": 1,
    "link": [
        {
            "relation": "self",
            "url": "http://192.168.1.62/healthconnect/cdr/fhir/r4/Patient?identifier=https%3A%2F%2Ffhir.nhs.uk%2FId%2Fnhs-number%7C9737873858"
        }
    ],
    "entry": [
        {
            "fullUrl": "http://192.168.1.62/healthconnect/cdr/fhir/r4/Patient/21370",
            "resource": {
                "resourceType": "Patient",
                "birthDate": "1986-09-12",
                "gender": "male",
                "id": "21370",
                "identifier": [
                    {
                        "assigner": {
                            "identifier": {
                                "system": "https://fhir.nhs.uk/Id/ods-orga

In [16]:
import json
import fhirclient.models.servicerequest as sr
import fhirclient.models.meta as meta
import pandas as pd
from dateutil import parser

def issued(issued):
    if issued == None:
        return None
    return parser.parse(issued.isostring)

def lastUpdated(meta : meta.Meta):
    if meta == None:
        return None
    return parser.parse(meta.lastUpdated.isostring)

if patientId != None:
    api_url = server + "ServiceRequest?patient="+patientId
    response = requests.get(api_url, auth=HTTPBasicAuth(fhir_username, fhir_password))
    response1JSON = response.json()

    print(response1JSON['total'])

    print(json.dumps(response1JSON, indent=4))

    serviceRequests = []
    for entry in response1JSON['entry']:
        #print(entry['resource'])
        print(entry['resource']['resourceType'], entry['resource']['status'])

        order = (sr.ServiceRequest(entry['resource']))
        serviceRequests.append(order)

    dfSR = pd.DataFrame([vars(s) for s in serviceRequests])
    ##dfDR['issued'] = dfDR['issued'].apply(issued)
    #dfDR['effectiveDateTime'] = dfDR['effectiveDateTime'].apply(issued)
    #dfDR['lastUpdated'] = dfDR['meta'].apply(lastUpdated)

dfSR

3
{
    "resourceType": "Bundle",
    "id": "81aa3db8-77d6-4712-aaa5-8f20af86a8ef",
    "type": "searchset",
    "timestamp": "2026-07-22T10:47:38Z",
    "total": 3,
    "link": [
        {
            "relation": "self",
            "url": "http://192.168.1.62/healthconnect/cdr/fhir/r4/ServiceRequest?patient=21370"
        }
    ],
    "entry": [
        {
            "fullUrl": "http://192.168.1.62/healthconnect/cdr/fhir/r4/ServiceRequest/21376",
            "resource": {
                "resourceType": "ServiceRequest",
                "category": [
                    {
                        "coding": [
                            {
                                "code": "116148004",
                                "system": "http://snomed.info/sct"
                            }
                        ]
                    }
                ],
                "code": {
                    "coding": [
                        {
                            "code": "M4.14",
       

,asNeededBoolean,asNeededCodeableConcept,authoredOn,basedOn,bodySite,category,code,doNotPerform,encounter,identifier,...,extension,modifierExtension,text,id,implicitRules,language,meta,_server,_resolved,_owner
0,None,None,None,None,None,[<fhirclient.models.codeableconcept.CodeableCo...,<fhirclient.models.codeableconcept.CodeableCon...,None,None,[<fhirclient.models.identifier.Identifier obje...,...,None,None,None,21376,None,None,<fhirclient.models.meta.Meta object at 0x10cc9...,None,None,None
1,None,None,<fhirclient.models.fhirdatetime.FHIRDateTime o...,[<fhirclient.models.fhirreference.FHIRReferenc...,None,[<fhirclient.models.codeableconcept.CodeableCo...,<fhirclient.models.codeableconcept.CodeableCon...,None,None,[<fhirclient.models.identifier.Identifier obje...,...,None,None,None,21377,None,None,<fhirclient.models.meta.Meta object at 0x10cd3...,None,None,None
2,None,None,None,None,None,[<fhirclient.models.codeableconcept.CodeableCo...,<fhirclient.models.codeableconcept.CodeableCon...,None,None,[<fhirclient.models.identifier.Identifier obje...,...,None,None,None,21940,None,None,<fhirclient.models.meta.Meta object at 0x10cd3...,None,None,None
